# Jev against a foundation model on Azure AI Foundry

The same 26 questions, asked of the same sentences, two ways:

- **Jev**, TypeSafe AI's *System One* model, through `jev_classifier`, as in [notebook 01](01_classify_reviews_with_jev.ipynb).
- **A foundation model** on Azure AI Foundry (the reflection deployment that writes the reviews), through a zero-shot [DSPy](https://dspy.ai) program in `dspy_classifier`. Its signature is built from Jev's own question set, so both are asked exactly the same thing, and both see the sentence and its whole review, never the star rating or the customer.

They are compared on **speed**, **cost**, **agreement** with each other, and **accuracy** against what the generator intended (`review_truth`). See issue #19.

Two things to keep in mind when reading the results:

- Jev *measures* its confidences; the foundation model only *reports* them. They fill the same columns, but are not equally trustworthy.
- The foundation model is a reasoning model. Its hidden reasoning is billed as output tokens and dominates its time, and `REASONING_EFFORT` trades one for the other.

Run [notebook 01](01_classify_reviews_with_jev.ipynb) first: this one reuses its dataset.

In [ ]:
from pathlib import Path

import polars as pl
from dotenv import load_dotenv

from dspy_classifier import (
    Run,
    accuracy,
    agreement,
    classify_sentences,
    cost,
    disagreements,
    foundry_lm_from_env,
    speed,
    stratified_sample,
)
from dspy_classifier.comparison import load_run, save_run
from jev_classifier import classify_sentences as jev_classify_sentences
from retail_generator.estimate import (
    JEV_SECONDS_PER_SENTENCE,
    JEV_TOKENS_PER_SENTENCE,
    JEV_USD_PER_MILLION_INPUT,
)
from retail_model import read_dataset
from review_wrangler import load_reviews, product_catalogue, split_sentences

load_dotenv(Path.cwd().parent / ".env")
OUTPUT = Path.cwd().parent / "data" / "output" / "comparison"
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(200)

# The comparison: change these and rerun.
SAMPLE_SENTENCES = 300          # whole reviews, stratified by language and stars (300 gives ~370); None for all
CONCURRENCY = 8                 # the same for both classifiers, so the race is fair
REASONING_EFFORT = None         # None for the deployment's default, or "minimal" / "low" / "medium" / "high"
FOUNDRY_PRICES = (None, None)   # USD per million input and output tokens for the deployment, to cost it
RERUN = False                   # False reuses saved runs: rerunning the analysis never pays twice

## 1. The sample

Whole reviews, so a review's intended issue can be scored against everything found in it, stratified by the language it was written in and its star rating.

In [ ]:
sentences = split_sentences(load_reviews()).collect()
catalogue = product_catalogue()
tables = read_dataset()
truth = tables["review_truth"]
sample = sentences if SAMPLE_SENTENCES is None else stratified_sample(sentences, SAMPLE_SENTENCES)
print(f"{sample.height} sentences from {sample['review_id'].n_unique()} reviews, of {sentences.height:,} sentences in the dataset")
sample.group_by("text_language").agg(reviews=pl.col("review_id").n_unique(), sentences=pl.len()).sort("sentences", descending=True)

## 2. What it will cost

Jev's figures are measured. For the foundation model, the prompt is measured by rendering it, but its output tokens cannot be known in advance: a reasoning model decides how long to think. The run measures them.

In [ ]:
from dspy_classifier.comparison import prompt_tokens

jev_tokens = sample.height * JEV_TOKENS_PER_SENTENCE
foundry_prompt = prompt_tokens(sample, catalogue)
print(f"Jev:     ~{jev_tokens:,} input tokens, ~${jev_tokens / 1e6 * JEV_USD_PER_MILLION_INPUT:.2f}, "
      f"~{sample.height * JEV_SECONDS_PER_SENTENCE / CONCURRENCY / 60:.1f} min")
print(f"Foundry: ~{foundry_prompt:,} input tokens plus reasoning and answer tokens, measured by the run")

## 3. Run both, live

Neither run uses a cache, so both are timed fresh, on the same sentences, at the same concurrency. Each run is saved to `data/output/comparison/`, and `RERUN = False` reuses it.

In [ ]:
async def run(name, classify):
    saved = OUTPUT / f"{name}.parquet"
    if not RERUN and saved.exists():
        return load_run(saved)
    import time
    started = time.perf_counter()
    results = await classify()
    return save_run(Run(name, results, time.perf_counter() - started, CONCURRENCY), saved)

lm = foundry_lm_from_env(reasoning_effort=REASONING_EFFORT)
jev = await run("jev", lambda: jev_classify_sentences(sample, catalogue, concurrency=CONCURRENCY))
foundry = await run("foundry", lambda: classify_sentences(sample, catalogue, lm=lm, concurrency=CONCURRENCY))
print(f"Foundry model: {lm.model}, reasoning effort: {REASONING_EFFORT or 'deployment default'}")

## 4. Speed

Latency per sentence, throughput at the same concurrency, and how long the whole dataset would take at that rate.

In [ ]:
speed([jev, foundry], sentences_in_dataset=sentences.height)

## 5. Cost

Tokens per sentence and dollars, scaled up to 1,000 customers. Jev charges for input tokens only. Set `FOUNDRY_PRICES` to cost the foundation model.

In [ ]:
sentences_per_1000 = round(sentences.height * 1000 / tables["customers"].height)
cost([jev, foundry], {"jev": (JEV_USD_PER_MILLION_INPUT, 0.0), "foundry": FOUNDRY_PRICES},
     sentences_per_1000_customers=sentences_per_1000)

## 6. Agreement

Per question, on the sentences both answered. Choices agree on the same label, yes/no questions on the same side of the 0.5 threshold, frustration within one level of its 0–4 rubric.

In [ ]:
agreement(jev.results, foundry.results)

In [ ]:
# Where they disagree about the problem
disagreements(jev.results, foundry.results, sample, "problem_category", names=("jev", "foundry"), n=10)

In [ ]:
# ...and about the sentiment
disagreements(jev.results, foundry.results, sample, "sentiment", names=("jev", "foundry"), n=10)

## 7. Accuracy against the truth

Neither classifier saw `review_truth`. Scored against it:

- **language:** the share of sentences heard in the language their review was written in (Zulu, which neither offers, should be `other`);
- **issue_found:** of the unhappy reviews, the share where the issue the review was written about is among the problems found in it;
- **primary_issue:** of those, the share where it is the problem in the review's most frustrated sentence.

In [ ]:
pl.DataFrame([{"classifier": r.name, **accuracy(r.results, sample, truth)} for r in (jev, foundry)])